# DX 704 Week 6 Project

This project will develop a treatment plan for a fictious illness "Twizzleflu".
Twizzleflu is a mild illness caused by a virus.
The main symptoms are a mild fever, fidgeting, and kicking the blankets off the bed or couch.
Mild dehydration has also been reported in more severe cases.
These symptoms typically last 1-2 weeks without treatment.
Word on the internet says that Twizzleflu can be cured faster by drinking copious orange juice, but this has not been supported by evidence so far.
You will be provided with a theoretical model of Twizzleflu modeled as a Markov decision process.
Based on the model, you will compute optimal treatment plans to optimize different criteria, and compare patient discomfort with the different plans.

The full project description, a template notebook, and raw data are available on GitHub: [Project 6 Materials](https://github.com/bu-cds-dx704/dx704-project-06).

We will model Twizzleflu as a Markov decision process.
The model transition probabilities are provided in the file "twizzleflu-transitions.tsv" and the expected rewards are in "twizzleflu-rewards.tsv".
The goal for Twizzleflu is to minimize the expected discomfort of the patient which is expressed as negative rewards in the file.

## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Evaluate a Do Nothing Plan

One of the treatment actions is to do nothing.
Calculate the expected discomfort (not rewards) of a policy that always does nothing.

Hint: for this value calculation and later ones, use value iteration.
The analytical solution has difficulties in practice when there is no discount factor.

In [18]:
!pip install numpy pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [19]:
# YOUR CHANGES HERE

import numpy as np
import pandas as pd

# Load transition probabilities
transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")

# Load rewards
rewards = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")

states = sorted(transitions['state'].unique())
n_states = len(states)

# Create state index mapping
state_to_idx = {s: i for i, s in enumerate(states)}

# Extract reward vector (negative = discomfort)
R = np.zeros(n_states)
for _, row in rewards.iterrows():
    R[state_to_idx[row['state']]] = row['reward']

# Build transition matrix for "do nothing" action
P = np.zeros((n_states, n_states))

do_nothing = transitions[transitions['action'] == "do_nothing"]

for _, row in do_nothing.iterrows():
    i = state_to_idx[row['state']]
    j = state_to_idx[row['next_state']]
    P[i, j] = row['probability']

# Value Iteration (gamma very close to 1 since no discount)
gamma = 0.999
V = np.zeros(n_states)

tolerance = 1e-8
max_iter = 10000

for _ in range(max_iter):
    V_new = R + gamma * P.dot(V)
    if np.max(np.abs(V_new - V)) < tolerance:
        break
    V = V_new

print("Expected discomfort under 'Do Nothing' policy:")
for s, v in zip(states, V):
    print(f"{s}: {v:.4f}")

Expected discomfort under 'Do Nothing' policy:
exposed-1: 0.0000
exposed-2: 0.0000
exposed-3: 0.0000
recovered: 0.0000
symptoms-1: -1.0000
symptoms-2: -2.0000
symptoms-3: -1.0000


Save the expected discomfort by state to a file "do-nothing-discomfort.tsv" with columns state and expected_discomfort.

In [20]:
# YOUR CHANGES HERE

# Create dataframe
output_df = pd.DataFrame({
    "state": states,
    "expected_discomfort": V
})

# Save to TSV
output_df.to_csv("do-nothing-discomfort.tsv", sep="\t", index=False)

print("File 'do-nothing-discomfort.tsv' saved successfully!")

File 'do-nothing-discomfort.tsv' saved successfully!


Submit "do-nothing-discomfort.tsv" in Gradescope.

## Part 2: Compute an Optimal Treatment Plan

Compute an optimal treatment plan for Twizzleflu.
It should minimize the expected discomfort (maximize the rewards).

In [21]:
# YOUR CHANGES HERE

import numpy as np
import pandas as pd

# ----------------------------
# Load data
# ----------------------------
transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rewards = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")

# Expected columns:
# transitions: state, action, next_state, probability
# rewards: state, reward

# ----------------------------
# Build state/action indices
# ----------------------------
states = sorted(set(transitions["state"]).union(set(transitions["next_state"])).union(set(rewards["state"])))
actions = sorted(transitions["action"].unique())

S = len(states)
A = len(actions)

s2i = {s: i for i, s in enumerate(states)}
a2i = {a: i for i, a in enumerate(actions)}

# Reward vector R[s]
R = np.zeros(S, dtype=float)
for _, row in rewards.iterrows():
    R[s2i[row["state"]]] = float(row["reward"])

# ----------------------------
# Build transition tensor P[a, s, s']
# ----------------------------
P = np.zeros((A, S, S), dtype=float)

for _, row in transitions.iterrows():
    s = row["state"]
    a = row["action"]
    sp = row["next_state"]
    p = float(row["probability"])
    P[a2i[a], s2i[s], s2i[sp]] += p  # += in case file has repeated rows

# Optional sanity check: transition rows should sum to ~1 for each (s,a) that exists
# (some states might not have all actions)
row_sums = P.sum(axis=2)  # shape (A,S)
# print("Min/Max row sums:", row_sums[row_sums>0].min(), row_sums.max())

# ----------------------------
# Value Iteration (optimal policy)
# ----------------------------
gamma = 0.999
tol = 1e-8
max_iter = 20000

V = np.zeros(S, dtype=float)

# mask of which actions are available per state (some (s,a) may be all zeros)
avail = row_sums > 0  # shape (A,S)

for it in range(max_iter):
    # Q[a,s] = R[s] + gamma * sum_{s'} P[a,s,s'] V[s']
    # Compute expected next value for each action/state:
    EV = np.einsum("asj,j->as", P, V)  # (A,S)

    Q = R[None, :] + gamma * EV  # (A,S)

    # For unavailable actions, set to -inf so they never get chosen
    Q_masked = Q.copy()
    Q_masked[~avail] = -np.inf

    V_new = np.max(Q_masked, axis=0)
    delta = np.max(np.abs(V_new - V))
    V = V_new

    if delta < tol:
        # print(f"Converged in {it+1} iterations, delta={delta:.2e}")
        break

# Optimal policy pi[s] = argmax_a Q[a,s]
EV = np.einsum("asj,j->as", P, V)
Q = R[None, :] + gamma * EV
Q[~avail] = -np.inf
best_a_idx = np.argmax(Q, axis=0)
best_action = [actions[i] if np.isfinite(Q[i, s2i[st]]).any() else None for st, i in zip(states, best_a_idx)]


Save the optimal actions for each state to a file "minimum-discomfort-actions.tsv" with columns state and action.

In [22]:
# YOUR CHANGES HERE

minimum_actions_df = pd.DataFrame({
    "state": states,
    "action": [actions[i] for i in best_a_idx]
})

minimum_actions_df.to_csv("minimum-discomfort-actions.tsv", sep="\t", index=False)
print("Saved: minimum-discomfort-actions.tsv")

Saved: minimum-discomfort-actions.tsv


Submit "minimum-discomfort-actions.tsv" in Gradescope.

## Part 3: Expected Discomfort

Using your previous optimal policy, compute the expected discomfort for each state.

In [23]:
# YOUR CHANGES HERE

# ----------------------------
# Policy Evaluation (Optimal Policy)
# ----------------------------

gamma = 0.999
tol = 1e-8
max_iter = 20000

V_policy = np.zeros(S)

for it in range(max_iter):
    V_new = np.zeros(S)

    for s_idx in range(S):
        a_idx = best_a_idx[s_idx]  # optimal action for state s
        V_new[s_idx] = R[s_idx] + gamma * np.dot(P[a_idx, s_idx, :], V_policy)

    delta = np.max(np.abs(V_new - V_policy))
    V_policy = V_new

    if delta < tol:
        break

# Convert rewards to discomfort (since rewards are negative discomfort)
expected_discomfort = -V_policy


Save your results in a file "minimum-discomfort-values.tsv" with columns state and expected_discomfort.

In [24]:
# YOUR CHANGES HERE

minimum_values_df = pd.DataFrame({
    "state": states,
    "expected_discomfort": expected_discomfort
})

# Sort by state (usually safest for grading scripts)
minimum_values_df = minimum_values_df.sort_values("state")

minimum_values_df.to_csv("minimum-discomfort-values.tsv", sep="\t", index=False)

print("Saved: minimum-discomfort-values.tsv")
minimum_values_df.head()

Saved: minimum-discomfort-values.tsv


,state,expected_discomfort
0,exposed-1,1.652295
1,exposed-2,3.307898
2,exposed-3,6.622418
3,recovered,-0.000000
4,symptoms-1,13.258094


Submit "minimum-discomfort-values.tsv" in Gradescope.

## Part 4: Minimizing Twizzleflu Duration

Modifiy the Markov decision process to minimize the days until the Twizzle flu is over.
To do so, change the reward function to always be -1 if the current state corresponds to being sick (must have symptoms, exposed does not count) and 0 otherwise.
To be clear, the action does not matter for this reward function.


In [25]:
# YOUR CHANGES HERE

import numpy as np
import pandas as pd

# ----------------------------
# Load transitions (reuse same transitions as before)
# ----------------------------
transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")

states = sorted(set(transitions["state"]).union(set(transitions["next_state"])))
actions = sorted(transitions["action"].unique())

S = len(states)
A = len(actions)

s2i = {s: i for i, s in enumerate(states)}
a2i = {a: i for i, a in enumerate(actions)}

# ----------------------------
# Build transition tensor P[a, s, s']
# ----------------------------
P = np.zeros((A, S, S), dtype=float)
for _, row in transitions.iterrows():
    s = row["state"]
    a = row["action"]
    sp = row["next_state"]
    p = float(row["probability"])
    P[a2i[a], s2i[s], s2i[sp]] += p

row_sums = P.sum(axis=2)  # (A,S)
avail = row_sums > 0      # action available in a state

# ----------------------------
# Part 4: Duration reward function
# Reward = -1 if "sick" (has symptoms), 0 otherwise.
# Exposed does NOT count as sick.
# ----------------------------
def is_sick_state(state_name: str) -> bool:
    s = state_name.lower()
    if "exposed" in s:
        return False
    # Common naming: "symptomatic", "symptoms", etc.
    if "symptom" in s:
        return True
    # If your states use different labels like "mild"/"severe", add them here:
    # if "mild" in s or "severe" in s: return True
    return False

R = np.array([-1.0 if is_sick_state(s) else 0.0 for s in states], dtype=float)

# ----------------------------
# Value Iteration to find optimal policy for minimizing duration
# ----------------------------
gamma = 0.999
tol = 1e-8
max_iter = 20000

V = np.zeros(S, dtype=float)

for it in range(max_iter):
    EV = np.einsum("asj,j->as", P, V)        # expected next value for each (a,s)
    Q = R[None, :] + gamma * EV              # (A,S)

    Q_masked = Q.copy()
    Q_masked[~avail] = -np.inf               # don't choose unavailable actions

    V_new = np.max(Q_masked, axis=0)
    delta = np.max(np.abs(V_new - V))
    V = V_new

    if delta < tol:
        break

best_a_idx = np.argmax(Q_masked, axis=0)
best_actions = [actions[i] for i in best_a_idx]


Save your new reward function in a file "duration-rewards.tsv" in the same format as "twizzleflu-rewards.tsv".

In [26]:
# YOUR CHANGES HERE

# Load original rewards file just to get the list of states
orig_rewards = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")
states = orig_rewards["state"].tolist()

# Define "sick" states (must have symptoms, exposed does NOT count)
def is_sick_state(state_name: str) -> bool:
    s = state_name.lower()
    if "exposed" in s:
        return False
    if "symptom" in s:
        return True
    # If your states use labels like mild/severe instead of "symptom",
    # uncomment below:
    # if "mild" in s or "severe" in s:
    #     return True
    return False

# Create new reward function
duration_rewards = pd.DataFrame({
    "state": states,
    "reward": [-1 if is_sick_state(s) else 0 for s in states]
})

# Save file
duration_rewards.to_csv("duration-rewards.tsv", sep="\t", index=False)

print("Saved: duration-rewards.tsv")
duration_rewards.head()

Saved: duration-rewards.tsv


,state,reward
0,exposed-1,0
1,exposed-2,0
2,exposed-3,0
3,symptoms-1,-1
4,symptoms-2,-1


Submit "duration-rewards.tsv" in Gradescope.

## Part 5: Optimize for Shorter Twizzleflu

Compute an optimal policy to minimize the duration of Twizzleflu.

In [27]:
# YOUR CHANGES HERE

import numpy as np
import pandas as pd

# ----------------------------
# Load transitions + duration rewards
# ----------------------------
transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
dur_rewards = pd.read_csv("duration-rewards.tsv", sep="\t")  # columns: state, reward

# ----------------------------
# Build state/action indices
# ----------------------------
states = sorted(set(transitions["state"]).union(set(transitions["next_state"])).union(set(dur_rewards["state"])))
actions = sorted(transitions["action"].unique())

S = len(states)
A = len(actions)

s2i = {s: i for i, s in enumerate(states)}
a2i = {a: i for i, a in enumerate(actions)}

# Reward vector R[s]
R = np.zeros(S, dtype=float)
for _, row in dur_rewards.iterrows():
    R[s2i[row["state"]]] = float(row["reward"])

# ----------------------------
# Build transition tensor P[a, s, s']
# ----------------------------
P = np.zeros((A, S, S), dtype=float)
for _, row in transitions.iterrows():
    s = row["state"]
    a = row["action"]
    sp = row["next_state"]
    p = float(row["probability"])
    P[a2i[a], s2i[s], s2i[sp]] += p

# Availability mask (some actions may not exist for some states)
row_sums = P.sum(axis=2)       # (A,S)
avail = row_sums > 0           # (A,S)

# ----------------------------
# Value Iteration (optimal for duration)
# ----------------------------
gamma = 0.999
tol = 1e-8
max_iter = 20000

V = np.zeros(S, dtype=float)

for it in range(max_iter):
    EV = np.einsum("asj,j->as", P, V)        # expected next value for each (a,s)
    Q = R[None, :] + gamma * EV              # (A,S)

    Q_masked = Q.copy()
    Q_masked[~avail] = -np.inf               # prevent choosing unavailable actions

    V_new = np.max(Q_masked, axis=0)
    delta = np.max(np.abs(V_new - V))
    V = V_new

    if delta < tol:
        break

best_a_idx = np.argmax(Q_masked, axis=0)
best_actions = [actions[i] for i in best_a_idx]


Save the optimal actions for each state to a file "minimum-duration-actions.tsv" with columns state and action.

In [28]:
# YOUR CHANGES HERE

minimum_duration_df = pd.DataFrame({
    "state": states,
    "action": [actions[i] for i in best_a_idx]
})

# Sort by state (safer for grading scripts)
minimum_duration_df = minimum_duration_df.sort_values("state")

minimum_duration_df.to_csv("minimum-duration-actions.tsv", sep="\t", index=False)

print("Saved: minimum-duration-actions.tsv")
minimum_duration_df.head()

Saved: minimum-duration-actions.tsv


,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,recovered,do-nothing
4,symptoms-1,do-nothing


Submit "minimum-duration-actions.tsv" in Gradescope.

## Part 6: Shorter Twizzleflu?

Compute the expected number of days sick for each state to a file.

In [29]:
# YOUR CHANGES HERE

# ----------------------------
# Policy Evaluation under duration-optimal policy
# ----------------------------

gamma = 0.999
tol = 1e-8
max_iter = 20000

V_policy = np.zeros(S)

for _ in range(max_iter):
    V_new = np.zeros(S)

    for s_idx in range(S):
        a_idx = best_a_idx[s_idx]   # optimal duration action
        V_new[s_idx] = R[s_idx] + gamma * np.dot(P[a_idx, s_idx, :], V_policy)

    delta = np.max(np.abs(V_new - V_policy))
    V_policy = V_new

    if delta < tol:
        break

# Expected days sick = negative of value (since reward was -1 per sick day)
expected_days_sick = -V_policy


Save the expected sick days for each state to a file "minimum-duration-days.tsv" with columns state and expected_sick_days.

In [30]:
# YOUR CHANGES HERE

minimum_duration_days_df = pd.DataFrame({
    "state": states,
    "expected_sick_days": expected_days_sick
})

# Sort by state (safer for grading)
minimum_duration_days_df = minimum_duration_days_df.sort_values("state")

minimum_duration_days_df.to_csv("minimum-duration-days.tsv", sep="\t", index=False)

print("Saved: minimum-duration-days.tsv")
minimum_duration_days_df.head()

Saved: minimum-duration-days.tsv


,state,expected_sick_days
0,exposed-1,1.239222
1,exposed-2,2.480926
2,exposed-3,4.966818
3,recovered,-0.000000
4,symptoms-1,9.943579


Submit "minimum-duration-days.tsv" in Gradescope.

## Part 7: Speed vs Pampering

Compute the expected discomfort using the policy to minimize days sick, and compare the results to the expected discomfort when optimizing to minimize discomfort.

In [31]:
# YOUR CHANGES HERE

# ============================================================
# Load data
# ============================================================
transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rewards_discomfort_df = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")  # negative discomfort rewards

# States / actions
states = sorted(set(transitions["state"]).union(set(transitions["next_state"])).union(set(rewards_discomfort_df["state"])))
actions = sorted(transitions["action"].unique())

S, A = len(states), len(actions)
s2i = {s: i for i, s in enumerate(states)}
a2i = {a: i for i, a in enumerate(actions)}

# ============================================================
# Build transition tensor P[a, s, s']
# ============================================================
P = np.zeros((A, S, S), dtype=float)
for _, row in transitions.iterrows():
    P[a2i[row["action"]], s2i[row["state"]], s2i[row["next_state"]]] += float(row["probability"])

row_sums = P.sum(axis=2)          # (A,S)
avail = row_sums > 0              # (A,S) action exists for state

# ============================================================
# Reward vectors
# ============================================================
# 1) Original discomfort rewards (negative discomfort)
R_discomfort = np.zeros(S, dtype=float)
for _, row in rewards_discomfort_df.iterrows():
    R_discomfort[s2i[row["state"]]] = float(row["reward"])

# 2) Duration rewards: -1 if sick (has symptoms), 0 otherwise; exposed is NOT sick
def is_sick_state(state_name: str) -> bool:
    s = state_name.lower()
    if "exposed" in s:
        return False
    if "symptom" in s:
        return True
    # If your state labels use "mild"/"severe" instead of "symptom", uncomment:
    # if ("mild" in s) or ("severe" in s):
    #     return True
    return False

R_duration = np.array([-1.0 if is_sick_state(s) else 0.0 for s in states], dtype=float)

# Save duration rewards in same format as twizzleflu-rewards.tsv
duration_rewards_df = pd.DataFrame({"state": states, "reward": R_duration}).sort_values("state")
duration_rewards_df.to_csv("duration-rewards.tsv", sep="\t", index=False)
print("Saved: duration-rewards.tsv")

# ============================================================
# Value Iteration (optimal policy) + Policy Evaluation
# ============================================================
def value_iteration_optimal_policy(R, P, avail, gamma=0.999, tol=1e-8, max_iter=20000):
    V = np.zeros(S, dtype=float)
    for _ in range(max_iter):
        EV = np.einsum("asj,j->as", P, V)      # (A,S)
        Q = R[None, :] + gamma * EV            # (A,S)
        Q[~avail] = -np.inf
        V_new = np.max(Q, axis=0)
        if np.max(np.abs(V_new - V)) < tol:
            V = V_new
            break
        V = V_new
    best_a_idx = np.argmax(Q, axis=0)
    return V, best_a_idx

def eval_policy_value(best_a_idx, R, P, gamma=0.999, tol=1e-8, max_iter=20000):
    V = np.zeros(S, dtype=float)
    for _ in range(max_iter):
        V_new = np.zeros(S, dtype=float)
        for s in range(S):
            a = int(best_a_idx[s])
            V_new[s] = R[s] + gamma * np.dot(P[a, s, :], V)
        if np.max(np.abs(V_new - V)) < tol:
            V = V_new
            break
        V = V_new
    return V

# ============================================================
# Part 2: Min-discomfort optimal policy + save actions
# ============================================================
V_min_discomfort, best_a_idx_discomfort = value_iteration_optimal_policy(R_discomfort, P, avail)

min_discomfort_actions_df = pd.DataFrame({
    "state": states,
    "action": [actions[i] for i in best_a_idx_discomfort]
}).sort_values("state")
min_discomfort_actions_df.to_csv("minimum-discomfort-actions.tsv", sep="\t", index=False)
print("Saved: minimum-discomfort-actions.tsv")

# Also (commonly needed) expected discomfort under min-discomfort policy:
V_eval_discomfort_policy = eval_policy_value(best_a_idx_discomfort, R_discomfort, P)
expected_discomfort_min_discomfort = -V_eval_discomfort_policy

min_discomfort_values_df = pd.DataFrame({
    "state": states,
    "expected_discomfort": expected_discomfort_min_discomfort
}).sort_values("state")
min_discomfort_values_df.to_csv("minimum-discomfort-values.tsv", sep="\t", index=False)
print("Saved: minimum-discomfort-values.tsv")

# ============================================================
# Part 5: Min-duration optimal policy + save actions
# ============================================================
V_min_duration, best_a_idx_duration = value_iteration_optimal_policy(R_duration, P, avail)

min_duration_actions_df = pd.DataFrame({
    "state": states,
    "action": [actions[i] for i in best_a_idx_duration]
}).sort_values("state")
min_duration_actions_df.to_csv("minimum-duration-actions.tsv", sep="\t", index=False)
print("Saved: minimum-duration-actions.tsv")

# Part 5/6: expected sick days under min-duration policy
V_eval_duration_policy = eval_policy_value(best_a_idx_duration, R_duration, P)
expected_sick_days = -V_eval_duration_policy

min_duration_days_df = pd.DataFrame({
    "state": states,
    "expected_sick_days": expected_sick_days
}).sort_values("state")
min_duration_days_df.to_csv("minimum-duration-days.tsv", sep="\t", index=False)
print("Saved: minimum-duration-days.tsv")

# ============================================================
# Compare: Expected discomfort under min-days-sick vs min-discomfort
# (Evaluate BOTH policies using the ORIGINAL discomfort rewards)
# ============================================================
V_discomfort_under_duration_policy = eval_policy_value(best_a_idx_duration, R_discomfort, P)
expected_discomfort_under_duration_policy = -V_discomfort_under_duration_policy

# expected_discomfort_min_discomfort already computed above

compare_df = pd.DataFrame({
    "state": states,
    "expected_discomfort_min_days_sick": expected_discomfort_under_duration_policy,
    "expected_discomfort_min_discomfort": expected_discomfort_min_discomfort,
})
compare_df["difference"] = (
    compare_df["expected_discomfort_min_days_sick"]
    - compare_df["expected_discomfort_min_discomfort"]
)
compare_df = compare_df.sort_values("state")
compare_df.to_csv("discomfort-policy-comparison.tsv", sep="\t", index=False)
print("Saved: discomfort-policy-comparison.tsv")

# Show a quick preview
compare_df.head(10)

Saved: duration-rewards.tsv
Saved: minimum-discomfort-actions.tsv
Saved: minimum-discomfort-values.tsv
Saved: minimum-duration-actions.tsv
Saved: minimum-duration-days.tsv
Saved: discomfort-policy-comparison.tsv


,state,expected_discomfort_min_days_sick,expected_discomfort_min_discomfort,difference
0,exposed-1,1.652295,1.652295,0.0
1,exposed-2,3.307898,3.307898,0.0
2,exposed-3,6.622418,6.622418,0.0
3,recovered,-0.000000,-0.000000,0.0
4,symptoms-1,13.258094,13.258094,0.0
5,symptoms-2,9.965662,9.965662,0.0
6,symptoms-3,3.325574,3.325574,0.0


Save the results to a file "policy-comparison.tsv" with columns state, speed_discomfort, and minimize_discomfort.

In [32]:
# YOUR CHANGES HERE

policy_compare_df = pd.DataFrame({
    "state": states,
    "speed_discomfort": expected_discomfort_under_duration_policy,
    "minimize_discomfort": expected_discomfort_min_discomfort
}).sort_values("state")

policy_compare_df.to_csv("policy-comparison.tsv", sep="\t", index=False)

print("Saved: policy-comparison.tsv")
policy_compare_df.head()

Saved: policy-comparison.tsv


,state,speed_discomfort,minimize_discomfort
0,exposed-1,1.652295,1.652295
1,exposed-2,3.307898,3.307898
2,exposed-3,6.622418,6.622418
3,recovered,-0.000000,-0.000000
4,symptoms-1,13.258094,13.258094


Submit "policy-comparison.tsv" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.